In [ ]:
from agents import Agent, Runner, function_tool, ItemHelpers

@function_tool
def get_weather(city: str):
    """Get weather by ciry"""
    return "-5 degrees"


agent = Agent(
    name="Assistant Agent",
    instructions="You ara helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather]
)

# runner는 while 루프를 실행해 주는 함수, 즉 ai 에이전트와 데이터를 주고 받으며 내가 원하는 응답을 줌
stream = Runner.run_streamed(agent, "Hello how are you? What is the weather in the capital of korea?")

async for event in stream.stream_events():
    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print("Agent updated to", event.new_agent.name)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict())
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item))
    print("=" * 20)

Agent updated to Assistant Agent
{'arguments': '{"city":"Seoul"}', 'call_id': 'call_0ZKeFlYWorXUpgUNanA3NHii', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_08b742243db1940b006986e2c62460819a832ecca191d5c8d7', 'status': 'completed'}
-5 degrees
Hello! I'm here to help. The weather in Seoul, the capital of South Korea, is currently -5 degrees Celsius. If you need more details or have other questions, feel free to ask!


In [ ]:
from agents import Agent, Runner, function_tool, ItemHelpers

@function_tool
def get_weather(city: str):
    """Get weather by ciry"""
    return "-5 degrees"


agent = Agent(
    name="Assistant Agent",
    instructions="You ara helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather]
)

stream = Runner.run_streamed(agent, "Hello how are you? What is the weather in the capital of korea?")

message = ""
args = ""

async for event in stream.stream_events():
    if event.type == "raw_response_event":
        event_type = event.data.type
        if event_type == "response.output_text.delta":
            message += event.data.delta
            print(message)
        elif event_type == "response.function_call_arguments.delta":
            args += event.data.delta
            print(args)
        elif event_type == "response.completed":
            message = ""
            args = ""

{"
{"city
{"city":"
{"city":"Se
{"city":"Seoul
{"city":"Seoul"}
Hello
Hello!
Hello! I'm
Hello! I'm doing
Hello! I'm doing well
Hello! I'm doing well,
Hello! I'm doing well, thank
Hello! I'm doing well, thank you
Hello! I'm doing well, thank you for
Hello! I'm doing well, thank you for asking
Hello! I'm doing well, thank you for asking.
Hello! I'm doing well, thank you for asking. The
Hello! I'm doing well, thank you for asking. The weather
Hello! I'm doing well, thank you for asking. The weather in
Hello! I'm doing well, thank you for asking. The weather in Seoul
Hello! I'm doing well, thank you for asking. The weather in Seoul,
Hello! I'm doing well, thank you for asking. The weather in Seoul, the
Hello! I'm doing well, thank you for asking. The weather in Seoul, the capital
Hello! I'm doing well, thank you for asking. The weather in Seoul, the capital of
Hello! I'm doing well, thank you for asking. The weather in Seoul, the capital of South
Hello! I'm doing well, thank you for asking

In [2]:
from agents import Agent, Runner, function_tool, SQLiteSession

session = SQLiteSession("user_2", "ai-memory.db")

@function_tool
def get_weather(city: str):
    """Get weather by ciry"""
    return "-5 degrees"


agent = Agent(
    name="Assistant Agent",
    instructions="You ara helpful assistant. Use tools when needed to answer questions",
    tools=[get_weather]
)

In [6]:
result = await Runner.run(
    agent, 
    "What was my name again?",
    session=session
)

print(result.final_output)

Your name is Pong.


In [7]:
await session.pop_item()

{'id': 'msg_075d4e31658c07e2006986eb8efab08198846335241a6a92ba',
 'content': [{'annotations': [],
   'text': 'Your name is Pong.',
   'type': 'output_text',
   'logprobs': []}],
 'role': 'assistant',
 'status': 'completed',
 'type': 'message'}

In [ ]:
from agents import Agent, Runner, function_tool, SQLiteSession, trace
from agents.extensions.visualization import draw_graph
from pydantic import BaseModel

session = SQLiteSession("user_1111", "ai-memory.db")

class Answer(BaseModel):
    answer: str
    background_explanation: str

@function_tool
def get_weather(city: str):
    """Get weather by ciry"""
    return "-5 degrees"


geaography_agent = Agent(
    name="Geo Expert Agent",
    instructions="You are a expert in geography, you answer questions related to them.",
    # mian_agent에게 이 agent가 어떤 역할을 하는지 알려주는 설명문
    handoff_description="Use this to answer geography related questions.",
    tools=[get_weather],
    output_type=Answer
)

economics_agent = Agent(
    name="Economics Expert Agent",
    instructions="You are a expert in economics, you answer questions related to them.",
    handoff_description="Use this to answer economics questions.",
)

main_agent = Agent(
    name="Main Agent",
    instructions="You are a user facing agent. Transfer to the agent most capable of answering the user's question.",
    # sub agent가 있다고 알려주는 기능
    handoffs=[
        economics_agent,
        geaography_agent,
    ],
)

# draw_graph(main_agent)

In [5]:
with trace("user_1111"):
    result = await Runner.run(
        main_agent, 
        "What is the capital of Colombia's northan province.",
        session=session
    )
    result = await Runner.run(
        main_agent, 
        "What is the capital of Cambodia's northan province.",
        session=session
    )
    result = await Runner.run(
        main_agent, 
        "What is the capital of Thailand's northan province.",
        session=session
    )